In [1]:
# 1. 경로 설정 + 주요 결과 파일 로드

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path(r"C:\취준\mobile-game-retention-review-ua-analysis")

OUTPUT_TABLE_DIR = PROJECT_DIR / "outputs" / "tables"
OUTPUT_FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
DOCS_DIR = PROJECT_DIR / "docs"
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"

for path in [OUTPUT_TABLE_DIR, OUTPUT_FIGURE_DIR, DOCS_DIR, NOTEBOOK_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def read_csv_safe(filename):
    path = OUTPUT_TABLE_DIR / filename
    if not path.exists():
        print(f"[MISSING] {filename}")
        return pd.DataFrame()
    print(f"[LOAD] {filename}")
    return pd.read_csv(path)

# Part 1. Cookie Cats A/B Test
cookie_group = read_csv_safe("cookie_cats_group_summary.csv")
cookie_ztest = read_csv_safe("cookie_cats_proportion_ztest_results.csv")
cookie_bootstrap = read_csv_safe("cookie_cats_bootstrap_retention_diff.csv")

# Part 2. Google Play Review Signal
review_final = read_csv_safe("google_play_review_final_summary.csv")
review_signal_summary = read_csv_safe("google_play_review_signal_summary.csv")
ad_comparison = read_csv_safe("google_play_ad_complaint_score_comparison.csv")
app_risk_score = read_csv_safe("google_play_app_review_risk_score.csv")

# Part 3. Campaign KPI / ROAS / ML
channel_kpi = read_csv_safe("02_channel_cpi_cvr.csv")
campaign_score = read_csv_safe("campaign_performance_score.csv")
budget_action_summary = read_csv_safe("campaign_budget_action_summary.csv")
model_results = read_csv_safe("d7_roas_model_results.csv")
error_by_channel = read_csv_safe("d7_roas_error_by_channel.csv")
error_by_country = read_csv_safe("d7_roas_error_by_country.csv")
error_by_creative = read_csv_safe("d7_roas_error_by_creative.csv")

print("\nLoaded files check complete.")

[LOAD] cookie_cats_group_summary.csv
[LOAD] cookie_cats_proportion_ztest_results.csv
[LOAD] cookie_cats_bootstrap_retention_diff.csv
[LOAD] google_play_review_final_summary.csv
[LOAD] google_play_review_signal_summary.csv
[LOAD] google_play_ad_complaint_score_comparison.csv
[LOAD] google_play_app_review_risk_score.csv
[LOAD] 02_channel_cpi_cvr.csv
[LOAD] campaign_performance_score.csv
[LOAD] campaign_budget_action_summary.csv
[LOAD] d7_roas_model_results.csv
[LOAD] d7_roas_error_by_channel.csv
[LOAD] d7_roas_error_by_country.csv
[LOAD] d7_roas_error_by_creative.csv

Loaded files check complete.


In [2]:
# 2. Part 1 요약: Cookie Cats A/B 테스트 핵심 결과 정리

cookie_summary_rows = []

if not cookie_group.empty and not cookie_ztest.empty:
    gate30 = cookie_group[cookie_group["version"] == "gate_30"].iloc[0]
    gate40 = cookie_group[cookie_group["version"] == "gate_40"].iloc[0]
    
    d1_test = cookie_ztest[cookie_ztest["metric"] == "retention_1"].iloc[0]
    d7_test = cookie_ztest[cookie_ztest["metric"] == "retention_7"].iloc[0]
    
    cookie_summary_rows.append({
        "part": "Part 1. A/B Test",
        "topic": "D1 Retention",
        "main_result": f"gate_30 {gate30['d1_retention']*100:.2f}%, gate_40 {gate40['d1_retention']*100:.2f}%",
        "difference": f"{d1_test['diff_gate40_minus_gate30']*100:.2f}%p",
        "statistical_result": f"p-value {d1_test['p_value']:.4f}",
        "interpretation": "gate_40은 D1 Retention에서 유의수준 5% 기준 명확한 개선을 보이지 못했다."
    })
    
    cookie_summary_rows.append({
        "part": "Part 1. A/B Test",
        "topic": "D7 Retention",
        "main_result": f"gate_30 {gate30['d7_retention']*100:.2f}%, gate_40 {gate40['d7_retention']*100:.2f}%",
        "difference": f"{d7_test['diff_gate40_minus_gate30']*100:.2f}%p",
        "statistical_result": f"p-value {d7_test['p_value']:.4f}",
        "interpretation": "gate_40은 D7 Retention이 유의하게 낮아, 실험안 적용에 신중해야 한다."
    })

cookie_final_summary = pd.DataFrame(cookie_summary_rows)
display(cookie_final_summary)

cookie_final_summary.to_csv(
    OUTPUT_TABLE_DIR / "final_cookie_cats_ab_test_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

,part,topic,main_result,difference,statistical_result,interpretation
0,Part 1. A/B Test,D1 Retention,"gate_30 44.82%, gate_40 44.23%",-0.59%p,p-value 0.0744,gate_40은 D1 Retention에서 유의수준 5% 기준 명확한 개선을 보이지...
1,Part 1. A/B Test,D7 Retention,"gate_30 19.02%, gate_40 18.20%",-0.82%p,p-value 0.0016,"gate_40은 D7 Retention이 유의하게 낮아, 실험안 적용에 신중해야 한다."


In [3]:
# 3. Part 2 요약: Google Play 리뷰 신호 분석 핵심 결과 정리

review_summary_rows = []

if not review_final.empty:
    r = review_final.iloc[0]
    
    review_summary_rows.append({
        "part": "Part 2. Review Signal",
        "topic": "Google Play 리뷰 수집 규모",
        "main_result": f"앱 {int(r['total_apps']):,}개, 리뷰 {int(r['total_reviews']):,}건",
        "difference": "-",
        "statistical_result": "-",
        "interpretation": "캐주얼/하이퍼캐주얼 게임 리뷰를 직접 수집해 앱마켓 유저 경험 신호 분석 기반을 구축했다."
    })
    
    review_summary_rows.append({
        "part": "Part 2. Review Signal",
        "topic": "광고 불만 리뷰",
        "main_result": f"광고 불만 리뷰 {int(r['ad_complaint_review_count']):,}건, 비율 {r['ad_complaint_review_rate']*100:.2f}%",
        "difference": f"광고 불만 포함 {r['ad_complaint_avg_score']:.2f}점 vs 미포함 {r['non_ad_complaint_avg_score']:.2f}점",
        "statistical_result": "-",
        "interpretation": "광고 불만이 포함된 리뷰는 평균 점수가 낮아, 광고 수익형 게임에서 광고 경험이 만족도 리스크로 나타날 수 있다."
    })
    
    review_summary_rows.append({
        "part": "Part 2. Review Signal",
        "topic": "리스크 신호 리뷰",
        "main_result": f"리스크 신호 포함 리뷰 비율 {r['risk_signal_review_rate']*100:.2f}%",
        "difference": "-",
        "statistical_result": "-",
        "interpretation": "광고, 오류, 반복성, 과금 등 리뷰 신호를 통해 앱마켓에서 관찰 가능한 UX 리스크를 구조화했다."
    })

if not review_signal_summary.empty:
    risk_only = review_signal_summary[
        review_signal_summary["signal_category"] != "positive_fun"
    ].copy()
    
    if not risk_only.empty:
        strongest_negative = risk_only.sort_values("score_diff_matched_minus_unmatched").iloc[0]
        review_summary_rows.append({
            "part": "Part 2. Review Signal",
            "topic": "가장 큰 점수 하락 신호",
            "main_result": strongest_negative["display_name"],
            "difference": f"{strongest_negative['score_diff_matched_minus_unmatched']:.2f}점",
            "statistical_result": "-",
            "interpretation": "해당 신호가 포함된 리뷰는 미포함 리뷰보다 평균 점수가 낮아, 주요 UX 리스크 후보로 해석할 수 있다."
        })

review_final_summary_table = pd.DataFrame(review_summary_rows)
display(review_final_summary_table)

review_final_summary_table.to_csv(
    OUTPUT_TABLE_DIR / "final_google_play_review_signal_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

,part,topic,main_result,difference,statistical_result,interpretation
0,Part 2. Review Signal,Google Play 리뷰 수집 규모,"앱 100개, 리뷰 9,761건",-,-,캐주얼/하이퍼캐주얼 게임 리뷰를 직접 수집해 앱마켓 유저 경험 신호 분석 기반을 구...
1,Part 2. Review Signal,광고 불만 리뷰,"광고 불만 리뷰 896건, 비율 9.18%",광고 불만 포함 2.58점 vs 미포함 4.21점,-,"광고 불만이 포함된 리뷰는 평균 점수가 낮아, 광고 수익형 게임에서 광고 경험이 만..."
2,Part 2. Review Signal,리스크 신호 리뷰,리스크 신호 포함 리뷰 비율 18.82%,-,-,"광고, 오류, 반복성, 과금 등 리뷰 신호를 통해 앱마켓에서 관찰 가능한 UX 리스..."
3,Part 2. Review Signal,가장 큰 점수 하락 신호,과금/IAP,-1.76점,-,"해당 신호가 포함된 리뷰는 미포함 리뷰보다 평균 점수가 낮아, 주요 UX 리스크 후..."


In [4]:
# 4. Part 3 요약: 캠페인 KPI/ROAS/ML 핵심 결과 정리

campaign_summary_rows = []

if not channel_kpi.empty:
    best_roas_channel = channel_kpi.sort_values("d7_roas", ascending=False).iloc[0]
    worst_roas_channel = channel_kpi.sort_values("d7_roas", ascending=True).iloc[0]
    
    campaign_summary_rows.append({
        "part": "Part 3. Campaign KPI/ROAS/ML",
        "topic": "채널별 D7 ROAS",
        "main_result": f"최고 {best_roas_channel['channel']} {best_roas_channel['d7_roas']:.4f}, 최저 {worst_roas_channel['channel']} {worst_roas_channel['d7_roas']:.4f}",
        "difference": "-",
        "statistical_result": "-",
        "interpretation": "채널별 유입 비용뿐 아니라 유입 이후 수익성 차이를 함께 확인해야 한다."
    })
    
    if "cpi" in channel_kpi.columns:
        lowest_cpi_channel = channel_kpi.sort_values("cpi", ascending=True).iloc[0]
        campaign_summary_rows.append({
            "part": "Part 3. Campaign KPI/ROAS/ML",
            "topic": "CPI와 ROAS의 차이",
            "main_result": f"최저 CPI 채널: {lowest_cpi_channel['channel']} {lowest_cpi_channel['cpi']:.2f}",
            "difference": f"D7 ROAS {lowest_cpi_channel['d7_roas']:.4f}",
            "statistical_result": "-",
            "interpretation": "CPI가 낮은 채널이 반드시 좋은 채널은 아니며, D7 Retention과 D7 ROAS를 함께 봐야 한다."
        })

if not budget_action_summary.empty:
    scale_row = budget_action_summary[budget_action_summary["budget_action"] == "Scale"]
    reduce_row = budget_action_summary[budget_action_summary["budget_action"] == "Reduce"]
    
    if not scale_row.empty:
        s = scale_row.iloc[0]
        campaign_summary_rows.append({
            "part": "Part 3. Campaign KPI/ROAS/ML",
            "topic": "Scale 후보",
            "main_result": f"{int(s['campaign_count'])}개 캠페인",
            "difference": f"평균 D7 ROAS {s['avg_d7_roas']:.4f}",
            "statistical_result": "-",
            "interpretation": "성과 점수가 높은 캠페인은 예산 확대 후보로 분류했다."
        })
    
    if not reduce_row.empty:
        red = reduce_row.iloc[0]
        campaign_summary_rows.append({
            "part": "Part 3. Campaign KPI/ROAS/ML",
            "topic": "Reduce 후보",
            "main_result": f"{int(red['campaign_count'])}개 캠페인",
            "difference": f"평균 D7 ROAS {red['avg_d7_roas']:.4f}",
            "statistical_result": "-",
            "interpretation": "성과 점수가 낮은 캠페인은 예산 축소 또는 소재/타겟 점검 후보로 분류했다."
        })

if not model_results.empty:
    best_model = model_results.sort_values("mae").iloc[0]
    campaign_summary_rows.append({
        "part": "Part 3. Campaign KPI/ROAS/ML",
        "topic": "D7 ROAS 예측 모델",
        "main_result": f"{best_model['model']} MAE {best_model['mae']:.4f}, R² {best_model['r2']:.3f}",
        "difference": "-",
        "statistical_result": "-",
        "interpretation": "모델 성능 자체를 과장하지 않고, 예측 오차가 큰 세그먼트를 확인해 Feature Engineering 개선 방향을 도출하는 데 초점을 두었다."
    })

campaign_final_summary = pd.DataFrame(campaign_summary_rows)
display(campaign_final_summary)

campaign_final_summary.to_csv(
    OUTPUT_TABLE_DIR / "final_campaign_kpi_roas_ml_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

,part,topic,main_result,difference,statistical_result,interpretation
0,Part 3. Campaign KPI/ROAS/ML,채널별 D7 ROAS,"최고 Unity Ads 0.0647, 최저 TikTok 0.0354",-,-,채널별 유입 비용뿐 아니라 유입 이후 수익성 차이를 함께 확인해야 한다.
1,Part 3. Campaign KPI/ROAS/ML,CPI와 ROAS의 차이,최저 CPI 채널: TikTok 4.71,D7 ROAS 0.0354,-,"CPI가 낮은 채널이 반드시 좋은 채널은 아니며, D7 Retention과 D7 R..."
2,Part 3. Campaign KPI/ROAS/ML,Scale 후보,3개 캠페인,평균 D7 ROAS 0.0757,-,성과 점수가 높은 캠페인은 예산 확대 후보로 분류했다.
3,Part 3. Campaign KPI/ROAS/ML,Reduce 후보,10개 캠페인,평균 D7 ROAS 0.0369,-,성과 점수가 낮은 캠페인은 예산 축소 또는 소재/타겟 점검 후보로 분류했다.
4,Part 3. Campaign KPI/ROAS/ML,D7 ROAS 예측 모델,"Linear Regression MAE 0.0097, R² 0.538",-,-,"모델 성능 자체를 과장하지 않고, 예측 오차가 큰 세그먼트를 확인해 Feature ..."


In [5]:
# 5. 최종 통합 인사이트 문서 생성

all_final_summary = pd.concat(
    [
        cookie_final_summary,
        review_final_summary_table,
        campaign_final_summary
    ],
    ignore_index=True
)

all_final_summary.to_csv(
    OUTPUT_TABLE_DIR / "final_integrated_project_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# 문서용 값 추출
cookie_d1_text = ""
cookie_d7_text = ""
if not cookie_final_summary.empty:
    d1_row = cookie_final_summary[cookie_final_summary["topic"] == "D1 Retention"].iloc[0]
    d7_row = cookie_final_summary[cookie_final_summary["topic"] == "D7 Retention"].iloc[0]
    cookie_d1_text = f"- D1 Retention: {d1_row['main_result']} / 차이 {d1_row['difference']} / {d1_row['statistical_result']}"
    cookie_d7_text = f"- D7 Retention: {d7_row['main_result']} / 차이 {d7_row['difference']} / {d7_row['statistical_result']}"

review_text = ""
if not review_final.empty:
    r = review_final.iloc[0]
    review_text = f"""
- 수집 앱 수: {int(r['total_apps']):,}개
- 수집 리뷰 수: {int(r['total_reviews']):,}건
- 광고 불만 리뷰 비율: {r['ad_complaint_review_rate']*100:.2f}%
- 광고 불만 포함 리뷰 평균 점수: {r['ad_complaint_avg_score']:.2f}
- 광고 불만 미포함 리뷰 평균 점수: {r['non_ad_complaint_avg_score']:.2f}
- 리스크 신호 포함 리뷰 비율: {r['risk_signal_review_rate']*100:.2f}%
"""

campaign_text = ""
if not channel_kpi.empty and not model_results.empty:
    best_roas_channel = channel_kpi.sort_values("d7_roas", ascending=False).iloc[0]
    worst_roas_channel = channel_kpi.sort_values("d7_roas", ascending=True).iloc[0]
    best_model = model_results.sort_values("mae").iloc[0]
    campaign_text = f"""
- 최고 D7 ROAS 채널: {best_roas_channel['channel']} ({best_roas_channel['d7_roas']:.4f})
- 최저 D7 ROAS 채널: {worst_roas_channel['channel']} ({worst_roas_channel['d7_roas']:.4f})
- D7 ROAS 예측 모델: {best_model['model']} / MAE {best_model['mae']:.4f} / R² {best_model['r2']:.3f}
"""

final_doc = f"""
# 모바일 게임 리텐션·리뷰·UA 성과 분석 최종 인사이트

## 1. 프로젝트 개요

이 프로젝트는 모바일 게임 데이터 분석·마케팅 분석 직무 지원을 위해 수행한 미니 포트폴리오 프로젝트다.

분석은 세 개의 독립 파트로 구성했다.

1. Cookie Cats 공개 A/B 테스트 데이터 기반 리텐션 분석
2. Google Play 게임 앱 메타데이터·리뷰 수집 기반 유저 경험 신호 분석
3. 보조 캠페인 데이터 기반 UA KPI·ROAS·ML 예측 오차 분석

서로 다른 출처와 기준 단위를 가진 데이터를 무리하게 결합하지 않고, 각 파트를 독립적으로 분석한 뒤 최종 인사이트에서만 모바일 게임 UA 성과 판단에 필요한 시사점으로 연결했다.

---

## 2. Part 1. Cookie Cats A/B 테스트 리텐션 분석

### 핵심 결과

{cookie_d1_text}
{cookie_d7_text}

### 해석

gate 위치를 level 30에서 level 40으로 늦춘 실험안은 D1 Retention에서 명확한 개선을 보이지 못했고, D7 Retention에서는 오히려 유의하게 낮은 결과를 보였다.

따라서 모바일 게임 A/B 테스트에서는 단기 반응만으로 실험안을 적용하기보다, D7 Retention처럼 더 긴 잔존 지표를 함께 확인해야 한다.

---

## 3. Part 2. Google Play 리뷰 신호 분석

### 핵심 결과

{review_text}

### 해석

Google Play 리뷰 분석 결과, 광고 불만이 포함된 리뷰는 광고 불만이 없는 리뷰보다 평균 점수가 낮게 나타났다.

이는 광고 수익형 게임에서 광고 노출 방식이 유저 경험과 만족도에 영향을 줄 수 있음을 시사한다. 다만 이 리뷰 데이터는 앱마켓 리뷰 데이터이므로 특정 UA 캠페인 성과와 직접 연결하지 않고, 앱 단위 유저 경험 리스크 신호로 제한 해석했다.

---

## 4. Part 3. 보조 캠페인 데이터 기반 UA KPI·ROAS·ML 분석

### 핵심 결과

{campaign_text}

### 해석

보조 캠페인 데이터 분석에서는 채널별 CPI, CVR, D7 Retention, D7 ROAS를 비교했다. 분석 결과 CPI가 낮은 채널이 반드시 좋은 성과를 보이는 것은 아니며, 유입 이후 잔존율과 수익성을 함께 봐야 했다.

또한 Campaign Performance Score를 산출해 캠페인을 Scale, Maintain, Review, Reduce 후보로 분류했다. 이 점수는 실제 광고 네트워크 최적화 알고리즘이 아니라, 캠페인 성과 비교를 위한 규칙 기반 스코어다. 다만 여러 지표를 표준화하고 가중합으로 결합하는 과정은 향후 ROAS/LTV 예측 모델의 Feature Engineering으로 확장 가능하다.

D7 ROAS 예측 모델은 초기 KPI로 단기 수익성을 어느 정도 예측할 수 있음을 보여주되, 모델 성능 자체보다 예측 오차가 큰 채널·국가·소재 유형을 확인하는 데 초점을 두었다.

---

## 5. 종합 인사이트

### Insight 1. 모바일 게임 실험은 단기 반응보다 리텐션 기준으로 판단해야 한다.

Cookie Cats A/B 테스트에서 D1 Retention은 명확한 차이를 보이지 않았지만, D7 Retention에서는 실험군이 유의하게 낮았다. 이는 게임 실험의 적용 여부를 판단할 때 단기 활동량이나 1일 잔존만 보는 것이 아니라, 더 긴 잔존 지표를 함께 확인해야 함을 보여준다.

### Insight 2. 광고 경험은 앱마켓 리뷰에서 명확한 불만 신호로 나타날 수 있다.

Google Play 리뷰 분석에서 광고 불만 키워드가 포함된 리뷰는 평균 점수가 낮게 나타났다. 이는 광고 수익형 게임에서 광고 노출 방식이 유저 경험 리스크로 이어질 수 있음을 시사한다.

### Insight 3. UA 캠페인은 CPI만으로 판단하면 안 된다.

보조 캠페인 분석에서는 낮은 CPI가 반드시 높은 D7 ROAS로 이어지지 않았다. 따라서 캠페인 성과는 CPI, CVR, D7 Retention, D7 ROAS를 함께 고려해야 한다.

### Insight 4. 예측 모델은 성능보다 오차 원인 분석까지 연결해야 한다.

D7 ROAS 예측 모델은 초기 KPI 기반으로 단기 성과를 예측하는 구조를 보여줬지만, 실제 실무에서는 예측이 빗나가는 세그먼트를 분석하는 것이 중요하다. 채널·국가·소재 유형별 오차 분석은 Feature Engineering 개선과 캠페인 운영 전략으로 이어질 수 있다.

---

## 6. 한계 및 개선 방향

| 한계 | 설명 | 개선 방향 |
|---|---|---|
| 데이터 간 직접 결합 없음 | A/B 테스트, 리뷰, 캠페인 데이터는 출처와 기준 단위가 달라 JOIN하지 않음 | 실제 게임사 내부 데이터 확보 시 유저·캠페인·리뷰·수익 로그 연결 가능 |
| 리뷰 분석의 대표성 한계 | Google Play 최신 리뷰 표본 기반 분석이며 전체 유저를 대표하지 않음 | 기간 확장, 국가별 리뷰 비교, 토픽 모델링/감성분석 고도화 |
| 캠페인 데이터의 보조성 | 실제 광고 데이터가 아니라 UA 분석 구조를 재현하기 위한 보조 데이터 | 실제 광고 집행 데이터 확보 시 ROAS, LTV, Incrementality 분석 가능 |
| ML 모델의 제한 | D7 ROAS 예측은 보조 데이터 기반이며 실제 운영 모델이 아님 | 실제 캠페인 로그 기반 D30 LTV 예측, calibration, 세그먼트별 모델 비교로 확장 |
| Incrementality 제한 | Cookie Cats A/B 테스트는 retention 실험이며 광고 채널 incremental lift 실험은 아님 | holdout/geo experiment 기반 광고 기여도 평가로 확장 |

---

## 7. 포트폴리오 메시지

이 프로젝트는 실제 기업 내부 광고 데이터를 사용한 프로젝트가 아니다. 대신 실제 공개 A/B 테스트 데이터, 직접 수집한 앱마켓 리뷰 데이터, 보조 캠페인 데이터를 명확히 분리해 분석했다.

핵심은 데이터를 억지로 결합하는 것이 아니라, 모바일 게임 성과 판단에서 필요한 리텐션, 유저 경험 신호, UA KPI, ROAS, 예측 오차 분석의 사고 흐름을 보여주는 것이다.
"""

(DOCS_DIR / "final_insights.md").write_text(final_doc, encoding="utf-8")

print("Saved:")
print(OUTPUT_TABLE_DIR / "final_integrated_project_summary.csv")
print(DOCS_DIR / "final_insights.md")

display(all_final_summary)

Saved:
C:\취준\mobile-game-retention-review-ua-analysis\outputs\tables\final_integrated_project_summary.csv
C:\취준\mobile-game-retention-review-ua-analysis\docs\final_insights.md


,part,topic,main_result,difference,statistical_result,interpretation
0,Part 1. A/B Test,D1 Retention,"gate_30 44.82%, gate_40 44.23%",-0.59%p,p-value 0.0744,gate_40은 D1 Retention에서 유의수준 5% 기준 명확한 개선을 보이지...
1,Part 1. A/B Test,D7 Retention,"gate_30 19.02%, gate_40 18.20%",-0.82%p,p-value 0.0016,"gate_40은 D7 Retention이 유의하게 낮아, 실험안 적용에 신중해야 한다."
2,Part 2. Review Signal,Google Play 리뷰 수집 규모,"앱 100개, 리뷰 9,761건",-,-,캐주얼/하이퍼캐주얼 게임 리뷰를 직접 수집해 앱마켓 유저 경험 신호 분석 기반을 구...
3,Part 2. Review Signal,광고 불만 리뷰,"광고 불만 리뷰 896건, 비율 9.18%",광고 불만 포함 2.58점 vs 미포함 4.21점,-,"광고 불만이 포함된 리뷰는 평균 점수가 낮아, 광고 수익형 게임에서 광고 경험이 만..."
4,Part 2. Review Signal,리스크 신호 리뷰,리스크 신호 포함 리뷰 비율 18.82%,-,-,"광고, 오류, 반복성, 과금 등 리뷰 신호를 통해 앱마켓에서 관찰 가능한 UX 리스..."
5,Part 2. Review Signal,가장 큰 점수 하락 신호,과금/IAP,-1.76점,-,"해당 신호가 포함된 리뷰는 미포함 리뷰보다 평균 점수가 낮아, 주요 UX 리스크 후..."
6,Part 3. Campaign KPI/ROAS/ML,채널별 D7 ROAS,"최고 Unity Ads 0.0647, 최저 TikTok 0.0354",-,-,채널별 유입 비용뿐 아니라 유입 이후 수익성 차이를 함께 확인해야 한다.
7,Part 3. Campaign KPI/ROAS/ML,CPI와 ROAS의 차이,최저 CPI 채널: TikTok 4.71,D7 ROAS 0.0354,-,"CPI가 낮은 채널이 반드시 좋은 채널은 아니며, D7 Retention과 D7 R..."
8,Part 3. Campaign KPI/ROAS/ML,Scale 후보,3개 캠페인,평균 D7 ROAS 0.0757,-,성과 점수가 높은 캠페인은 예산 확대 후보로 분류했다.
9,Part 3. Campaign KPI/ROAS/ML,Reduce 후보,10개 캠페인,평균 D7 ROAS 0.0369,-,성과 점수가 낮은 캠페인은 예산 축소 또는 소재/타겟 점검 후보로 분류했다.
